In [1]:
import yaml
import json
import os

from src.datasets.dataset_from_tg import data_full_routine
from src.transformers.transformers_utils import init_pretrained_model, tokenize_function

/mnt/z/projects/code/llm-on-telegram-chats/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
CONFIG_DIR = "configs/tune"
CONFIG_NAME = "rut5_small.test.yaml" # parse arg
CONFIG_PATH = os.path.join(CONFIG_DIR, CONFIG_NAME)

In [3]:
with open(CONFIG_PATH, "r") as f:
    config = yaml.safe_load(f)
    
random_seed = config["RANDOM_SEED"]
model_name = config["MODEL_NAME"]

data_path = os.path.join(config["DATA_DIR"], config["DATA_NAME"])
with open(data_path, "r") as f:
    raw_data = json.load(f)

name = config["RESPONSE_NAME"]

In [22]:
# model, tokenizer = init_pretrained_model(model_name, random_seed, device_map="cuda:0")
from transformers import T5ForConditionalGeneration, T5Tokenizer

tokenizer = T5Tokenizer.from_pretrained("cointegrated/rut5-small-chitchat")
model = T5ForConditionalGeneration.from_pretrained("cointegrated/rut5-small-chitchat")

dataset = data_full_routine(raw_data, name)
# tokenized_dataset = dataset.map(lambda x: tokenize_function(sample=x, tokenizer=tokenizer), batched=True)

100%|██████████| 785/785 [00:00<00:00, 1548696.44it/s]


In [23]:
from datasets import Dataset
dataset = Dataset.from_dict(dataset[:10000])

In [24]:
train_test_split = dataset.train_test_split(test_size=0.3, shuffle=True, )
data_train = train_test_split["train"]
data_eval = train_test_split["test"]

In [25]:
def tokenize_function(examples):
    return tokenizer(examples['context'], padding="max_length", truncation=True)

def preprocess_function(examples):
    inputs = [f"dialogue: {context} </s>" for context in examples["context"]]
    targets = [f"{response} </s>" for response in examples["response"]]
    model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding="max_length")

    # Настройка labels
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(targets, max_length=512, truncation=True, padding="max_length")

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

data_train = data_train.map(preprocess_function, batched=True)
data_eval = data_eval.map(preprocess_function, batched=True)

Map:   0%|          | 0/7000 [00:00<?, ? examples/s]/mnt/z/projects/code/llm-on-telegram-chats/venv/lib/python3.12/site-packages/transformers/models/t5/tokenization_t5.py:289: UserWarning: This sequence already has </s>. In future versions this behavior may lead to duplicated eos tokens being added.
  warnings.warn(
/mnt/z/projects/code/llm-on-telegram-chats/venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:3953: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(
Map: 100%|██████████| 3000/3000 [00:00<00:00, 3131.55 examples/s]


In [26]:
data_train = data_train.map(tokenize_function, batched=True)
data_eval = data_eval.map(tokenize_function, batched=True)

Map:   0%|          | 0/7000 [00:00<?, ? examples/s]Asking to pad to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no padding.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Map: 100%|██████████| 3000/3000 [00:00<00:00, 11735.17 examples/s]


In [27]:
model.eval()

T5ForConditionalGeneration(
  (shared): Embedding(20100, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(20100, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=384, bias=False)
              (k): Linear(in_features=512, out_features=384, bias=False)
              (v): Linear(in_features=512, out_features=384, bias=False)
              (o): Linear(in_features=384, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 6)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=512, out_features=1024, bias=False)
              (wi_1): Linear(in_features=512, out_features=1024, bias=False)
              (wo): 

In [28]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [29]:
data_eval["response"]

['Ну да прикинуть надо че то\nНо вроде вполне прикольная задачка\n\n',
 'ааа\nпфф\nя думал завалила это типа когда не сдал\n\n',
 'Ну да и ей лет 40 как я понял\n\n',
 'год и два месяца назад тебе рассказывал\n\n',
 'блин, вот щас подумал что учеба была тебе к лицу\n\n',
 'да тебе ж не удобно\n\n',
 'с лизой?\nда ну не\n\n',
 'Я б еще репы проебал\n\n',
 'В лицо мне это скажи блин\n\n',
 'ну там да могут приходы быть всякие\n\n',
 'ахах\n\n',
 'я рассматривал такой варик, когда в фаблабе пару занятий проводил мне даже понравилось\nно щас каким то нереалистичным аспирантура и преподавание кажутся\nтипо как то круто в науку сложно влезть из за того что мы на половину технологи и нет большого количества важных знаний\n+ даже норм диплома нет\nи слишком научная деятельность отрывается от того чем я на работе занимался\nи при этом работа вроде пока нравится и перспективы там более видимые\n\n',
 'Я завтра с ним спишусь, как ему будет удобнее\nПо идее к 14-15 смогу подъехать\nЗавтра точнее н

In [30]:
data_eval

Dataset({
    features: ['context', 'response', 'chat', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 3000
})

In [31]:
model

T5ForConditionalGeneration(
  (shared): Embedding(20100, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(20100, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=384, bias=False)
              (k): Linear(in_features=512, out_features=384, bias=False)
              (v): Linear(in_features=512, out_features=384, bias=False)
              (o): Linear(in_features=384, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 6)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=512, out_features=1024, bias=False)
              (wi_1): Linear(in_features=512, out_features=1024, bias=False)
              (wo): 

In [32]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./temp",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    num_train_epochs=5,
    weight_decay=0.01,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=data_train,
    eval_dataset=data_eval,  # Можно добавить валидационный датасет
    data_collator=data_collator,
)

trainer.train()

/mnt/z/projects/code/llm-on-telegram-chats/venv/lib/python3.12/site-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Epoch,Training Loss,Validation Loss
1,0.195400,0.183439
2,0.189800,0.178894
3,0.185800,0.176757
4,0.184500,0.175685
5,0.181600,0.175362


TrainOutput(global_step=8750, training_loss=0.45223201511928013, metrics={'train_runtime': 1796.6444, 'train_samples_per_second': 19.481, 'train_steps_per_second': 4.87, 'total_flos': 537928002772992.0, 'train_loss': 0.45223201511928013, 'epoch': 5.0})

In [33]:
SAVE_DIR = "artifacts/20241208_rut5_test"

model.save_pretrained(SAVE_DIR + "/model")
tokenizer.save_pretrained(SAVE_DIR + "/tokenizer")

('artifacts/20241208_rut5_test/tokenizer/tokenizer_config.json',
 'artifacts/20241208_rut5_test/tokenizer/special_tokens_map.json',
 'artifacts/20241208_rut5_test/tokenizer/spiece.model',
 'artifacts/20241208_rut5_test/tokenizer/added_tokens.json')

In [ ]:
model = T5ForConditionalGeneration.from_pretrained('path/to/save/model')
tokenizer = T5Tokenizer.from_pretrained('path/to/save/model')

In [45]:
input_text = "Жопич"
input_ids = tokenizer.encode(input_text, return_tensors='pt').to("cuda:0")
output_ids = model.generate(input_ids)
output_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)

In [46]:
system_prompt = "Ты Дима, с тобой общаются люди в чате. Тебе нужно отвечать на их вопросы."
full_input_text = f"{system_prompt} {input_text}"
input_ids = tokenizer.encode(full_input_text, return_tensors='pt').to("cuda:0")
output_ids = model.generate(input_ids)
output_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)

In [47]:
output_text

'ну я уже давно уже говорил что я уже давно'

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [11]:
X1 = np.linspace(-10, 10, 20)
X2 = np.linspace(-10, 10, 20)

In [15]:
Y = np.zeros(shape=(len(X1), len(X2)))
for row in range(Y.shape[0]):
    for col in range(Y.shape[1]):
        if row < 10:
            if col < 10:
                Y[row, col] = 1
            else:
                Y[row, col] = 0
        else:
            if col < 10:
                Y[row, col] = 0
            else:
                Y[row, col] = 1

print(Y)

[[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 1. 1. 1. 1. 1. 1. 1.

In [20]:
X1 * X2

array([100.        ,  80.05540166,  62.32686981,  46.81440443,
        33.51800554,  22.43767313,  13.5734072 ,   6.92520776,
         2.49307479,   0.27700831,   0.27700831,   2.49307479,
         6.92520776,  13.5734072 ,  22.43767313,  33.51800554,
        46.81440443,  62.32686981,  80.05540166, 100.        ])